# Week 4 TAIRO RL Benchmark Notebook

**Project:** TAIRO — Trustworthy AI Robotics Framework for Failure-Aware and Cybersecurity-Resilient Caring Robots  
**Focus:** Reinforcement learning benchmarking, attack simulation, dataset generation, baselines, and experimental design.

This notebook provides a starter implementation plan for:

1. Creating a small benchmark dataset from `FetchReach-v4`
2. Running clean and attacked robot-control episodes
3. Testing sensor-noise and action-manipulation attacks
4. Comparing random, rule-based, SAC, and SAC+HER-style baselines
5. Computing trustworthiness metrics
6. Exporting episode-level results for Week 4/Week 5 presentations

Primary benchmark: Gymnasium Robotics FetchReach-v4  
URL: https://robotics.farama.org/envs/fetch/reach/

Stable-Baselines3 SAC:  
https://stable-baselines3.readthedocs.io/en/master/modules/sac.html

Stable-Baselines3 HER:
https://stable-baselines3.readthedocs.io/en/master/modules/her.html


## 1. Installation Notes

Run the installation cell only if the required packages are not already installed.

For local machines, MuJoCo and Gymnasium Robotics may require additional system dependencies. If installation fails, use the synthetic dataset section first to test the analysis pipeline.


In [ ]:
# Optional installation cell.
# Uncomment and run if packages are missing.

# !pip install gymnasium gymnasium-robotics mujoco stable-baselines3 pandas numpy matplotlib tqdm

## 2. Imports and Global Settings

In [ ]:
import os
import math
import json
import random
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
    import gymnasium_robotics
    GYM_AVAILABLE = True
except Exception as e:
    print("Gymnasium Robotics is not available in this runtime.")
    print("You can still run the synthetic dataset and analysis sections.")
    print("Import error:", repr(e))
    GYM_AVAILABLE = False

try:
    from stable_baselines3 import SAC
    from stable_baselines3.her.her_replay_buffer import HerReplayBuffer
    SB3_AVAILABLE = True
except Exception as e:
    print("Stable-Baselines3 is not available in this runtime.")
    print("You can still run random/rule-based baselines and analysis sections.")
    print("Import error:", repr(e))
    SB3_AVAILABLE = False

RANDOM_SEEDS = [0, 1, 2]
ENV_ID = "FetchReach-v4"
MAX_EPISODE_STEPS = 50
RESULT_DIR = "tairo_results"
os.makedirs(RESULT_DIR, exist_ok=True)

np.random.seed(0)
random.seed(0)


AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


## 3. Experimental Design

The Week 4/Week 5 experiment should compare clean and attacked robot behavior.

### Main Conditions

| Condition | Description |
|---|---|
| `clean` | No perturbation |
| `sensor_noise` | Gaussian noise added to the observation vector |
| `action_noise` | Gaussian noise added to the action vector |
| `action_scale` | Action magnitude scaled up or down |
| `action_delay` | Previous action is executed instead of current action |
| `target_shift` | Desired goal is shifted during the episode |

### Main Baselines

| Method | Purpose |
|---|---|
| Random policy | Sanity-check baseline |
| Rule-based reaching controller | Transparent non-learning baseline |
| SAC | Standard continuous-control RL baseline |
| SAC+HER | Goal-conditioned RL baseline |
| Recovery-aware SAC+HER | Proposed TAIRO method with detection/recovery logic |


In [ ]:
experiment_design = pd.DataFrame([
    {
        "Experiment": "Clean baseline",
        "Condition": "clean",
        "Methods": "Random, Rule-based, SAC, SAC+HER",
        "Metrics": "Success rate, average reward, final distance, episode length"
    },
    {
        "Experiment": "Sensor-noise attack",
        "Condition": "sensor_noise",
        "Methods": "Rule-based, SAC+HER, Recovery-aware SAC+HER",
        "Metrics": "Success rate, reward drop, final distance, attack success rate"
    },
    {
        "Experiment": "Action-manipulation attack",
        "Condition": "action_noise / action_scale / action_delay",
        "Methods": "Rule-based, SAC+HER, Recovery-aware SAC+HER",
        "Metrics": "Success rate, recovery rate, safety violation rate, trustworthiness score"
    },
    {
        "Experiment": "Optional target shift",
        "Condition": "target_shift",
        "Methods": "Rule-based, SAC+HER, Recovery-aware SAC+HER",
        "Metrics": "Recovery success rate, time-to-recovery, success after shift"
    },
])

experiment_design


,Experiment,Condition,Methods,Metrics
0,Clean baseline,clean,"Random, Rule-based, SAC, SAC+HER","Success rate, average reward, final distance, ..."
1,Sensor-noise attack,sensor_noise,"Rule-based, SAC+HER, Recovery-aware SAC+HER","Success rate, reward drop, final distance, att..."
2,Action-manipulation attack,action_noise / action_scale / action_delay,"Rule-based, SAC+HER, Recovery-aware SAC+HER","Success rate, recovery rate, safety violation ..."
3,Optional target shift,target_shift,"Rule-based, SAC+HER, Recovery-aware SAC+HER","Recovery success rate, time-to-recovery, succe..."


## 4. Helper Functions for FetchReach-v4

FetchReach-v4 returns a dictionary observation with keys such as:

- `observation`
- `achieved_goal`
- `desired_goal`

The helper functions below extract features, compute distance-to-goal, and apply attacks.


In [ ]:
def make_env(seed: int = 0):
    """Create FetchReach-v4 environment if Gymnasium Robotics is available."""
    if not GYM_AVAILABLE:
        raise RuntimeError("Gymnasium Robotics is not available.")
    env = gym.make(ENV_ID, max_episode_steps=MAX_EPISODE_STEPS)
    env.reset(seed=seed)
    return env


def flatten_goal_obs(obs: Dict[str, np.ndarray]) -> np.ndarray:
    """Flatten goal-conditioned observation dictionary for logging or rule-based control."""
    return np.concatenate([
        np.asarray(obs["observation"], dtype=np.float32).ravel(),
        np.asarray(obs["achieved_goal"], dtype=np.float32).ravel(),
        np.asarray(obs["desired_goal"], dtype=np.float32).ravel()
    ])


def distance_to_goal(obs: Dict[str, np.ndarray]) -> float:
    """Euclidean distance between achieved_goal and desired_goal."""
    return float(np.linalg.norm(obs["achieved_goal"] - obs["desired_goal"]))


def add_sensor_noise(obs: Dict[str, np.ndarray], noise_std: float) -> Dict[str, np.ndarray]:
    """Add Gaussian noise to observation fields while preserving dictionary structure."""
    attacked = {}
    for key, value in obs.items():
        arr = np.asarray(value).copy()
        attacked[key] = arr + np.random.normal(loc=0.0, scale=noise_std, size=arr.shape)
    return attacked


def shift_target(obs: Dict[str, np.ndarray], shift_scale: float = 0.03) -> Dict[str, np.ndarray]:
    """Shift desired goal to simulate target spoofing or unexpected goal change."""
    attacked = {key: np.asarray(value).copy() for key, value in obs.items()}
    attacked["desired_goal"] = attacked["desired_goal"] + np.random.uniform(
        low=-shift_scale, high=shift_scale, size=attacked["desired_goal"].shape
    )
    return attacked


def manipulate_action(
    action: np.ndarray,
    attack_type: str = "none",
    noise_std: float = 0.05,
    scale: float = 1.0,
    previous_action: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Apply action-level attack."""
    action = np.asarray(action).copy()

    if attack_type == "none":
        attacked_action = action
    elif attack_type == "action_noise":
        attacked_action = action + np.random.normal(0.0, noise_std, size=action.shape)
    elif attack_type == "action_scale":
        attacked_action = scale * action
    elif attack_type == "action_reverse":
        attacked_action = -1.0 * action
    elif attack_type == "action_delay" and previous_action is not None:
        attacked_action = previous_action
    else:
        attacked_action = action

    return np.clip(attacked_action, -1.0, 1.0)


def action_smoothness(actions: List[np.ndarray]) -> float:
    """Lower values indicate smoother control. Computes mean action-difference norm."""
    if len(actions) < 2:
        return 0.0
    diffs = [np.linalg.norm(actions[i] - actions[i-1]) for i in range(1, len(actions))]
    return float(np.mean(diffs))


def action_magnitude(actions: List[np.ndarray]) -> float:
    """Mean action magnitude. Large values may indicate unstable or aggressive behavior."""
    if len(actions) == 0:
        return 0.0
    return float(np.mean([np.linalg.norm(a) for a in actions]))


## 5. Baseline Policies

### Random Policy
Samples a random action from the environment.

### Rule-Based Reaching Controller
Moves the end-effector toward the desired goal using the difference between `desired_goal` and `achieved_goal`.

This controller is intentionally simple and transparent. It is useful as a non-learning baseline.


In [ ]:
def random_policy(env, obs: Dict[str, np.ndarray]) -> np.ndarray:
    return env.action_space.sample()


def rule_based_reach_policy(env, obs: Dict[str, np.ndarray], gain: float = 5.0) -> np.ndarray:
    """
    Simple reaching controller for FetchReach.
    Action space is usually 4D: dx, dy, dz, gripper.
    FetchReach does not require gripper control, so gripper action is set to 0.
    """
    achieved = np.asarray(obs["achieved_goal"])
    desired = np.asarray(obs["desired_goal"])
    direction = desired - achieved

    action = np.zeros(env.action_space.shape, dtype=np.float32)
    action[:3] = gain * direction[:3]
    action[-1] = 0.0

    return np.clip(action, env.action_space.low, env.action_space.high)


## 6. Episode Runner

This function runs one episode under a specified method and attack condition.

Supported methods:

- `random`
- `rule_based`
- `sac_model` if a Stable-Baselines3 model is provided


In [ ]:
@dataclass
class EpisodeResult:
    method: str
    condition: str
    seed: int
    attack_level: float
    total_reward: float
    success: float
    final_distance: float
    episode_length: int
    action_smoothness: float
    action_magnitude: float
    safety_violation: float
    recovery_used: float


def run_episode(
    env,
    method: str,
    seed: int,
    condition: str = "clean",
    attack_level: float = 0.0,
    model=None,
    use_recovery: bool = False,
    target_shift_step: int = 25,
) -> Tuple[EpisodeResult, pd.DataFrame]:
    """Run one episode and return summary plus step-level log."""

    obs, info = env.reset(seed=seed)
    total_reward = 0.0
    terminated = False
    truncated = False
    step_logs = []
    actions = []
    previous_action = np.zeros(env.action_space.shape, dtype=np.float32)
    recovery_used = 0.0

    for t in range(MAX_EPISODE_STEPS):
        clean_obs = obs

        # Observation-level attacks
        policy_obs = clean_obs
        if condition == "sensor_noise":
            policy_obs = add_sensor_noise(clean_obs, noise_std=attack_level)
        elif condition == "target_shift" and t >= target_shift_step:
            policy_obs = shift_target(clean_obs, shift_scale=attack_level)

        # Select action
        if method == "random":
            action = random_policy(env, policy_obs)
        elif method == "rule_based":
            action = rule_based_reach_policy(env, policy_obs)
        elif method in {"sac", "sac_her", "recovery_aware_sac_her"} and model is not None:
            action, _ = model.predict(policy_obs, deterministic=True)
        else:
            # Fallback to rule-based controller if model is unavailable.
            action = rule_based_reach_policy(env, policy_obs)

        intended_action = np.asarray(action).copy()

        # Action-level attacks
        executed_action = intended_action
        if condition == "action_noise":
            executed_action = manipulate_action(intended_action, "action_noise", noise_std=attack_level)
        elif condition == "action_scale":
            executed_action = manipulate_action(intended_action, "action_scale", scale=1.0 + attack_level)
        elif condition == "action_reverse":
            executed_action = manipulate_action(intended_action, "action_reverse")
        elif condition == "action_delay":
            executed_action = manipulate_action(intended_action, "action_delay", previous_action=previous_action)

        # Simple recovery logic
        # If observation/action appears too risky, reduce action magnitude.
        if use_recovery:
            if np.linalg.norm(executed_action - intended_action) > 0.15 or distance_to_goal(policy_obs) > 0.25:
                executed_action = 0.5 * executed_action
                recovery_used = 1.0

        previous_action = executed_action.copy()
        actions.append(executed_action.copy())

        obs, reward, terminated, truncated, info = env.step(executed_action)
        total_reward += float(reward)

        current_distance = distance_to_goal(obs)
        is_success = float(info.get("is_success", 0.0))

        # Simple safety proxy: unusually large action or worsening distance.
        safety_violation_step = float(np.linalg.norm(executed_action) > 1.5)

        step_logs.append({
            "method": method,
            "condition": condition,
            "seed": seed,
            "attack_level": attack_level,
            "timestep": t,
            "reward": float(reward),
            "distance_to_goal": current_distance,
            "is_success": is_success,
            "action_norm": float(np.linalg.norm(executed_action)),
            "intended_action_norm": float(np.linalg.norm(intended_action)),
            "safety_violation": safety_violation_step,
            "recovery_used": recovery_used,
        })

        if terminated or truncated:
            break

    step_df = pd.DataFrame(step_logs)

    result = EpisodeResult(
        method=method,
        condition=condition,
        seed=seed,
        attack_level=float(attack_level),
        total_reward=float(total_reward),
        success=float(step_df["is_success"].max() if len(step_df) else 0.0),
        final_distance=float(step_df["distance_to_goal"].iloc[-1] if len(step_df) else np.nan),
        episode_length=int(len(step_df)),
        action_smoothness=action_smoothness(actions),
        action_magnitude=action_magnitude(actions),
        safety_violation=float(step_df["safety_violation"].max() if len(step_df) else 0.0),
        recovery_used=float(recovery_used),
    )

    return result, step_df


## 7. Generate Example Dataset from FetchReach-v4

This section runs a small experiment and saves:

- Episode-level data: `tairo_fetchreach_episode_results.csv`
- Step-level trajectory data: `tairo_fetchreach_step_logs.csv`

If Gymnasium Robotics is unavailable, skip to the synthetic dataset section.


In [ ]:
def run_small_fetchreach_benchmark():
    if not GYM_AVAILABLE:
        print("Skipping real FetchReach benchmark because Gymnasium Robotics is unavailable.")
        return pd.DataFrame(), pd.DataFrame()

    all_results = []
    all_steps = []

    methods = ["random", "rule_based"]
    conditions = [
        ("clean", 0.00),
        ("sensor_noise", 0.01),
        ("sensor_noise", 0.05),
        ("action_noise", 0.05),
        ("action_scale", 0.50),
        ("action_delay", 0.00),
        ("target_shift", 0.03),
    ]

    for seed in RANDOM_SEEDS:
        env = make_env(seed=seed)

        for method in methods:
            for condition, attack_level in conditions:
                result, step_df = run_episode(
                    env=env,
                    method=method,
                    seed=seed,
                    condition=condition,
                    attack_level=attack_level,
                    use_recovery=False,
                )
                all_results.append(asdict(result))
                all_steps.append(step_df)

                # Recovery-aware version for attack conditions
                if condition != "clean":
                    result_r, step_df_r = run_episode(
                        env=env,
                        method=method + "_recovery",
                        seed=seed,
                        condition=condition,
                        attack_level=attack_level,
                        use_recovery=True,
                    )
                    all_results.append(asdict(result_r))
                    all_steps.append(step_df_r)

        env.close()

    episode_df = pd.DataFrame(all_results)
    step_df = pd.concat(all_steps, ignore_index=True) if all_steps else pd.DataFrame()

    episode_path = os.path.join(RESULT_DIR, "tairo_fetchreach_episode_results.csv")
    step_path = os.path.join(RESULT_DIR, "tairo_fetchreach_step_logs.csv")

    episode_df.to_csv(episode_path, index=False)
    step_df.to_csv(step_path, index=False)

    print("Saved:", episode_path)
    print("Saved:", step_path)

    return episode_df, step_df


episode_df, step_df = run_small_fetchreach_benchmark()
episode_df.head() if len(episode_df) else pd.DataFrame()


## 8. Synthetic Dataset Example

Use this section if the real FetchReach environment is not available. The synthetic dataset follows the same structure as the real experiment output, so the metric and plotting sections still work.

This is useful for preparing Week 4 slides before the real experiments are complete.


In [ ]:
def generate_synthetic_tairo_results(num_seeds: int = 5) -> pd.DataFrame:
    rng = np.random.default_rng(42)

    methods = ["random", "rule_based", "sac", "sac_her", "recovery_aware_sac_her"]
    conditions = [
        ("clean", 0.00),
        ("sensor_noise", 0.01),
        ("sensor_noise", 0.05),
        ("sensor_noise", 0.10),
        ("action_noise", 0.05),
        ("action_scale", 0.50),
        ("action_delay", 0.00),
        ("target_shift", 0.03),
    ]

    rows = []
    for seed in range(num_seeds):
        for method in methods:
            for condition, attack_level in conditions:
                base_success = {
                    "random": 0.05,
                    "rule_based": 0.65,
                    "sac": 0.75,
                    "sac_her": 0.88,
                    "recovery_aware_sac_her": 0.90,
                }[method]

                attack_penalty = 0.0
                if condition == "sensor_noise":
                    attack_penalty = 1.5 * attack_level
                elif condition == "action_noise":
                    attack_penalty = 0.20
                elif condition == "action_scale":
                    attack_penalty = 0.15
                elif condition == "action_delay":
                    attack_penalty = 0.25
                elif condition == "target_shift":
                    attack_penalty = 0.18

                recovery_bonus = 0.15 if "recovery" in method and condition != "clean" else 0.0
                success_prob = np.clip(base_success - attack_penalty + recovery_bonus, 0.0, 1.0)
                success = float(rng.random() < success_prob)

                final_distance = float(np.clip(rng.normal(0.04 + attack_penalty * 0.2 - recovery_bonus * 0.1, 0.02), 0.0, 0.30))
                total_reward = float(-50 * final_distance + 10 * success + rng.normal(0, 1))
                smoothness = float(np.clip(rng.normal(0.2 + attack_penalty, 0.05), 0.0, 2.0))
                action_mag = float(np.clip(rng.normal(0.8 + attack_penalty, 0.10), 0.0, 3.0))
                safety_violation = float((smoothness > 0.55) or (action_mag > 1.25))
                recovery_used = float("recovery" in method and condition != "clean")

                rows.append({
                    "method": method,
                    "condition": condition,
                    "seed": seed,
                    "attack_level": attack_level,
                    "total_reward": total_reward,
                    "success": success,
                    "final_distance": final_distance,
                    "episode_length": int(rng.integers(20, 51)),
                    "action_smoothness": smoothness,
                    "action_magnitude": action_mag,
                    "safety_violation": safety_violation,
                    "recovery_used": recovery_used,
                })

    return pd.DataFrame(rows)


if len(episode_df) == 0:
    episode_df = generate_synthetic_tairo_results()
    episode_df.to_csv(os.path.join(RESULT_DIR, "synthetic_tairo_episode_results.csv"), index=False)
    print("Using synthetic results for analysis examples.")

episode_df.head()


## 9. Metric Computation

The trustworthiness score combines:

- Reliability
- Robustness
- Cybersecurity resilience
- Safety
- Recovery

For Week 4, use a simple equal-weight formula first:

```text
Trustworthiness Score =
0.20 × Reliability
+ 0.20 × Robustness
+ 0.20 × Cybersecurity Resilience
+ 0.20 × Safety
+ 0.20 × Recovery
```


In [ ]:
def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["method", "condition", "attack_level"]
    summary = (
        df.groupby(group_cols)
        .agg(
            success_rate=("success", "mean"),
            avg_reward=("total_reward", "mean"),
            final_distance=("final_distance", "mean"),
            avg_episode_length=("episode_length", "mean"),
            action_smoothness=("action_smoothness", "mean"),
            action_magnitude=("action_magnitude", "mean"),
            safety_violation_rate=("safety_violation", "mean"),
            recovery_rate=("recovery_used", "mean"),
        )
        .reset_index()
    )
    return summary


summary_df = summarize_results(episode_df)
summary_df


In [ ]:
def add_trustworthiness_scores(summary: pd.DataFrame) -> pd.DataFrame:
    out = summary.copy()

    # Reliability: success rate under the current condition.
    out["reliability"] = out["success_rate"].clip(0, 1)

    # Robustness: use final distance and smoothness as rough degradation proxies.
    # Smaller final distance and smoother actions are better.
    max_distance = max(out["final_distance"].max(), 1e-6)
    max_smoothness = max(out["action_smoothness"].max(), 1e-6)

    distance_score = 1 - (out["final_distance"] / max_distance)
    smoothness_score = 1 - (out["action_smoothness"] / max_smoothness)

    out["robustness"] = (0.6 * distance_score + 0.4 * smoothness_score).clip(0, 1)

    # Cybersecurity resilience: success under attack-like conditions.
    # Clean condition receives resilience equal to reliability.
    out["attack_success_rate"] = 1 - out["success_rate"]
    out["cybersecurity_resilience"] = (1 - out["attack_success_rate"]).clip(0, 1)

    # Safety: inverse of safety violation rate.
    out["safety"] = (1 - out["safety_violation_rate"]).clip(0, 1)

    # Recovery: recovery rate for attacked conditions; clean condition set to 1.0 by convention.
    out["recovery"] = out["recovery_rate"]
    out.loc[out["condition"] == "clean", "recovery"] = 1.0

    out["trustworthiness_score"] = (
        0.20 * out["reliability"]
        + 0.20 * out["robustness"]
        + 0.20 * out["cybersecurity_resilience"]
        + 0.20 * out["safety"]
        + 0.20 * out["recovery"]
    )

    return out


score_df = add_trustworthiness_scores(summary_df)
score_df.to_csv(os.path.join(RESULT_DIR, "tairo_summary_with_scores.csv"), index=False)
score_df


## 10. Visualization Examples

Use these plots in the Week 4 or Week 5 presentation.

The plots are intentionally simple and presentation-friendly.


In [ ]:
# Success rate by method and condition
plot_df = score_df.copy()
plot_df["method_condition"] = plot_df["method"] + "\n" + plot_df["condition"] + "\n" + plot_df["attack_level"].astype(str)

plt.figure(figsize=(12, 5))
plt.bar(plot_df["method_condition"], plot_df["success_rate"])
plt.xticks(rotation=90)
plt.ylabel("Success Rate")
plt.title("TAIRO Benchmark: Success Rate by Method and Condition")
plt.tight_layout()
plt.show()


In [ ]:
# Trustworthiness score by method and condition
plt.figure(figsize=(12, 5))
plt.bar(plot_df["method_condition"], plot_df["trustworthiness_score"])
plt.xticks(rotation=90)
plt.ylabel("Trustworthiness Score")
plt.title("TAIRO Benchmark: Trustworthiness Score by Method and Condition")
plt.tight_layout()
plt.show()


In [ ]:
# Final distance by method and condition
plt.figure(figsize=(12, 5))
plt.bar(plot_df["method_condition"], plot_df["final_distance"])
plt.xticks(rotation=90)
plt.ylabel("Final Distance to Goal")
plt.title("TAIRO Benchmark: Final Distance to Goal")
plt.tight_layout()
plt.show()


## 11. SAC+HER Training Template

This section is a template for training SAC+HER with Stable-Baselines3.

Depending on the installed Stable-Baselines3 version, HER may be configured through `HerReplayBuffer`.

This can take time. Start with a small number of timesteps for debugging, then increase.


In [ ]:
def train_sac_her_template(total_timesteps: int = 10_000, seed: int = 0):
    """
    Template for SAC+HER training on FetchReach-v4.
    This cell may require working Gymnasium Robotics, MuJoCo, and Stable-Baselines3.
    """
    if not GYM_AVAILABLE:
        raise RuntimeError("Gymnasium Robotics is not available.")
    if not SB3_AVAILABLE:
        raise RuntimeError("Stable-Baselines3 is not available.")

    env = make_env(seed=seed)

    model = SAC(
        policy="MultiInputPolicy",
        env=env,
        replay_buffer_class=HerReplayBuffer,
        replay_buffer_kwargs=dict(
            n_sampled_goal=4,
            goal_selection_strategy="future",
        ),
        verbose=1,
        seed=seed,
        learning_rate=1e-3,
        buffer_size=100_000,
        batch_size=256,
        gamma=0.95,
        tau=0.05,
    )

    model.learn(total_timesteps=total_timesteps)

    model_path = os.path.join(RESULT_DIR, "sac_her_fetchreach_model")
    model.save(model_path)
    env.close()

    print("Saved model to:", model_path)
    return model


# Uncomment to train.
# sac_her_model = train_sac_her_template(total_timesteps=10_000, seed=0)


## 12. Evaluate a Trained SAC+HER Model Under Attacks

After training or loading a model, use the same `run_episode` function to evaluate clean and attacked conditions.

This creates the same result format as the random and rule-based baselines.


In [ ]:
def evaluate_trained_model_template(model, seeds=RANDOM_SEEDS):
    if not GYM_AVAILABLE:
        raise RuntimeError("Gymnasium Robotics is not available.")

    results = []
    steps = []

    conditions = [
        ("clean", 0.00),
        ("sensor_noise", 0.01),
        ("sensor_noise", 0.05),
        ("action_noise", 0.05),
        ("action_scale", 0.50),
        ("action_delay", 0.00),
        ("target_shift", 0.03),
    ]

    for seed in seeds:
        env = make_env(seed=seed)

        for condition, attack_level in conditions:
            result, step_df = run_episode(
                env=env,
                method="sac_her",
                seed=seed,
                condition=condition,
                attack_level=attack_level,
                model=model,
                use_recovery=False,
            )
            results.append(asdict(result))
            steps.append(step_df)

            result_r, step_df_r = run_episode(
                env=env,
                method="recovery_aware_sac_her",
                seed=seed,
                condition=condition,
                attack_level=attack_level,
                model=model,
                use_recovery=True,
            )
            results.append(asdict(result_r))
            steps.append(step_df_r)

        env.close()

    model_episode_df = pd.DataFrame(results)
    model_step_df = pd.concat(steps, ignore_index=True)

    model_episode_df.to_csv(os.path.join(RESULT_DIR, "sac_her_attack_episode_results.csv"), index=False)
    model_step_df.to_csv(os.path.join(RESULT_DIR, "sac_her_attack_step_logs.csv"), index=False)

    return model_episode_df, model_step_df


# Example after training:
# model_episode_df, model_step_df = evaluate_trained_model_template(sac_her_model)
# model_episode_df.head()


## 13. Week 4 Slide Tables

Use the following tables directly in the Week 4 presentation.


In [ ]:
baseline_table = pd.DataFrame([
    {"Baseline": "Random policy", "Purpose": "Sanity-check baseline", "Expected Role": "Lowest expected performance"},
    {"Baseline": "Rule-based reaching controller", "Purpose": "Transparent non-learning baseline", "Expected Role": "Simple, interpretable comparison"},
    {"Baseline": "SAC", "Purpose": "Standard continuous-control RL baseline", "Expected Role": "Tests RL without HER"},
    {"Baseline": "SAC+HER", "Purpose": "Goal-conditioned RL baseline", "Expected Role": "Main RL comparison"},
    {"Baseline": "No-recovery policy", "Purpose": "Tests behavior without safety response", "Expected Role": "Shows value of recovery logic"},
    {"Baseline": "Recovery-aware SAC+HER", "Purpose": "TAIRO proposed method", "Expected Role": "Improves robustness and recovery"},
])
baseline_table


In [ ]:
benchmark_resource_table = pd.DataFrame([
    {
        "Benchmark": "Gymnasium Robotics",
        "URL": "https://robotics.farama.org/",
        "TAIRO Role": "Main robotics benchmark package"
    },
    {
        "Benchmark": "FetchReach-v4",
        "URL": "https://robotics.farama.org/envs/fetch/reach/",
        "TAIRO Role": "Primary Week 4 RL benchmark"
    },
    {
        "Benchmark": "Stable-Baselines3 SAC",
        "URL": "https://stable-baselines3.readthedocs.io/en/master/modules/sac.html",
        "TAIRO Role": "Continuous-control RL implementation"
    },
    {
        "Benchmark": "Stable-Baselines3 HER",
        "URL": "https://stable-baselines3.readthedocs.io/en/master/modules/her.html",
        "TAIRO Role": "Goal-conditioned RL replay strategy"
    },
    {
        "Benchmark": "Robust-Gymnasium",
        "URL": "https://github.com/SafeRL-Lab/Robust-Gymnasium",
        "TAIRO Role": "Robust RL disturbance benchmark reference"
    },
    {
        "Benchmark": "Habitat 3.0",
        "URL": "https://aihabitat.org/habitat3/",
        "TAIRO Role": "Future home/human-centered robotics extension"
    },
    {
        "Benchmark": "AI2-THOR",
        "URL": "https://ai2thor.allenai.org/",
        "TAIRO Role": "Future indoor embodied AI benchmark"
    },
    {
        "Benchmark": "World Labs Marble",
        "URL": "https://marble.worldlabs.ai/",
        "TAIRO Role": "Generated-world safety audit"
    },
])
benchmark_resource_table


## 14. Suggested Week 5 Implementation Checklist

1. Install and test `FetchReach-v4`
2. Run random policy and rule-based controller
3. Train or test SAC+HER
4. Add sensor-noise attack
5. Add action-manipulation attack
6. Run at least 3 random seeds
7. Save episode-level and step-level CSV files
8. Compute success rate, reward drop, final distance, safety violation rate, and trustworthiness score
9. Add simple recovery logic: halt, retry, or reduced action
10. Prepare one result table and one plot
11. Update GitHub with README, code, results, and benchmark URLs


## 15. Final Notes for Students

For Week 4, the most important contribution is not to install every simulator.

The strongest experimental path is:

```text
FetchReach-v4 + SAC+HER + sensor-noise attack + action-manipulation attack
+ random/rule-based baselines + trustworthiness score
```

Habitat, AI2-THOR, RoboTHOR, MiniGrid, and World Labs/Marble can be presented as future extensions after the FetchReach benchmark is working.
